# Temperaturas en Uluru y Wollongong

In [ ]:
# Instalación opcional en Google Colab
import sys

if "google.colab" in sys.modules:
    %pip install -q "pymc==6.4.0" "arviz-base==1.3.1" "matplotlib==3.11.2" "nutpie==0.16.11" polars

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pymc as pm

from pymc.model.fgraph import clone_model

plt.rcParams["axes.prop_cycle"] = plt.cycler(color=["#0072B2", "#E69F00", "#009E73", "#CC79A7"])
az.rcParams["plot.backend"] = "matplotlib"
az.rcParams["stats.ci_kind"] = "eti"
az.rcParams["stats.ci_prob"] = 0.95
SEMILLA = 2026

## Problema

El conjunto de datos `weather_WU` contiene datos climáticos diarios correspondientes a dos ciudades de Australia: Uluru y Wollongong. Dos de las variables que se encuentran son la temperatura a las 9 de la mañana (`temp9am`) y la temperatura a las 3 de la tarde (`temp3pm`).

Proponga un modelo que permita describir la asociación entre las temperaturas. A partir de su modelo:

- ¿Cuál es la probabilidad de que la temperatura a la tarde sea mayor a 30 °C cuando a la mañana hicieron 22 °C en Uluru? ¿Y en Wollongong?
- Para cada ciudad, ¿cuál es la probabilidad de que la temperatura a la tarde sea mayor a 30 °C independientemente de la temperatura de la mañana?
- Para cada ciudad, ¿cuál es la probabilidad de que la temperatura de la tarde sea menor que la de la mañana?

In [ ]:
url_weather = "https://raw.githubusercontent.com/tomicapretto/datos-sae-2025/refs/heads/main/datos/weather_WU.csv"
df_weather = pl.read_csv(url_weather).select("location", "temp9am", "temp3pm")
df_weather.head()

In [ ]:
df_weather.group_by("location").len().sort("location")

Tenemos 200 registros, 100 por ciudad. Las temperaturas están expresadas en °C. Veamos primero la relación entre la mañana y la tarde, distinguiendo las ciudades por color.

In [ ]:
x = df_weather["temp9am"].to_numpy()
y = df_weather["temp3pm"].to_numpy()
ciudades, ciudad_idx = np.unique(df_weather["location"].to_numpy(), return_inverse=True)
grupos = [0, 1]

fig, ax = plt.subplots(figsize=(7, 4))
for grupo, ciudad in zip(grupos, ciudades):
    seleccion = ciudad_idx == grupo
    ax.scatter(x[seleccion], y[seleccion], color=f"C{grupo}", s=24, alpha=0.7, label=ciudad)
ax.set(xlabel="Temperatura a las 9 (°C)", ylabel="Temperatura a las 15 (°C)")
ax.legend(title="Ciudad");

En ambas ciudades, las mañanas más cálidas suelen acompañarse de tardes más cálidas. Pero las dos nubes están desplazadas: para temperaturas parecidas a la mañana, Uluru suele tener tardes más cálidas. Veamos qué ocurre si empezamos usando una sola recta.

In [ ]:
coords = {"obs_idx": np.arange(len(y)), "ciudad": ciudades}
parametros = ["beta_0", "beta_1", "sigma"]
x_grid = np.linspace(5, 40, 50)
x_grid_ciudades = np.tile(x_grid, 2)
ciudad_grid = np.repeat([0, 1], len(x_grid))

## Modelo 1: una recta para ambas ciudades

Conservamos los _priors_ del curso anterior. Este primer modelo usa un intercepto, una pendiente y un desvío residual comunes.

In [ ]:
with pm.Model(coords=coords) as modelo_1:
    x_data = pm.Data("temp9am", x, dims="obs_idx")

    beta_0 = pm.Normal("beta_0", mu=10, sigma=5)
    beta_1 = pm.Normal("beta_1", mu=0, sigma=1)
    sigma = pm.HalfNormal("sigma", sigma=10)
    mu = pm.Deterministic("mu", beta_0 + beta_1 * x_data, dims="obs_idx")
    pm.Normal("temp3pm", mu=mu, sigma=sigma, observed=y, dims="obs_idx")

In [ ]:
with modelo_1:
    idata_1 = pm.sample(chains=4, random_seed=SEMILLA)

In [ ]:
az.summary(idata_1, var_names=parametros, ci_prob=0.95, round_to=2)

In [ ]:
with clone_model(modelo_1):
    pm.set_data({"temp9am": x_grid}, coords={"obs_idx": np.arange(len(x_grid))})
    preds_1 = pm.sample_posterior_predictive(
        idata_1, var_names=["mu", "temp3pm"], predictions=True, random_seed=SEMILLA
    )

mu_1 = az.extract(preds_1, group="predictions", var_names="mu").to_numpy().T
y_1 = az.extract(preds_1, group="predictions", var_names="temp3pm").to_numpy().T
mu_media_1 = mu_1.mean(axis=0)
mu_inferior_1, mu_superior_1 = np.quantile(mu_1, [0.025, 0.975], axis=0)
y_inferior_1, y_superior_1 = np.quantile(y_1, [0.025, 0.975], axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_grid, mu_media_1, color="0.4")
ax.fill_between(x_grid, y_inferior_1, y_superior_1, color="0.4", alpha=0.15)
ax.fill_between(x_grid, mu_inferior_1, mu_superior_1, color="0.4", alpha=0.45)

for grupo, ciudad in zip(grupos, ciudades):
    seleccion = ciudad_idx == grupo
    ax.scatter(x[seleccion], y[seleccion], color=f"C{grupo}", s=20, alpha=0.7, label=ciudad)

ax.set(xlabel="Temperatura a las 9 (°C)", ylabel="Temperatura a las 15 (°C)")
ax.legend(title="Ciudad");

La recta queda entre las dos nubes. Las observaciones de Uluru tienden a estar por encima y las de Wollongong por debajo. Para abarcar ambas, la banda predictiva resulta bastante amplia.

Generemos ahora réplicas de las temperaturas de la tarde para las mañanas que observamos y hagamos un chequeo predictivo _a posteriori_.

In [ ]:
with modelo_1:
    pm.sample_posterior_predictive(
        idata_1, var_names=["temp3pm"], extend_inferencedata=True, random_seed=SEMILLA
    )

In [ ]:
az.plot_ppc_dist(idata_1, var_names="temp3pm", num_samples=50, figure_kwargs={"figsize": (7, 3.5)})
ax = plt.gcf().axes[0]
ax.set(
    title="Modelo 1: Una recta para ambas ciudades",
    xlabel="Temperatura a las 15 (°C)",
    ylabel="Densidad",
);

## Modelo 2: un intercepto por ciudad

Ahora cada ciudad tiene su propio intercepto y conservamos una pendiente común. Los demás supuestos son los mismos.

In [ ]:
with pm.Model(coords=coords) as modelo_2:
    x_data = pm.Data("temp9am", x, dims="obs_idx")
    ciudad_data = pm.Data("ciudad_idx", ciudad_idx, dims="obs_idx")

    beta_0 = pm.Normal("beta_0", mu=10, sigma=5, dims="ciudad")
    beta_1 = pm.Normal("beta_1", mu=0, sigma=1)
    sigma = pm.HalfNormal("sigma", sigma=10)
    mu = pm.Deterministic("mu", beta_0[ciudad_data] + beta_1 * x_data, dims="obs_idx")
    pm.Normal("temp3pm", mu=mu, sigma=sigma, observed=y, dims="obs_idx")

In [ ]:
with modelo_2:
    idata_2 = pm.sample(chains=4, random_seed=SEMILLA)

In [ ]:
az.summary(idata_2, var_names=parametros, ci_prob=0.95, round_to=2)

In [ ]:
with clone_model(modelo_2):
    pm.set_data(
        {"temp9am": x_grid_ciudades, "ciudad_idx": ciudad_grid},
        coords={"obs_idx": np.arange(len(x_grid_ciudades))},
    )
    preds_2 = pm.sample_posterior_predictive(
        idata_2, var_names=["mu", "temp3pm"], predictions=True, random_seed=SEMILLA
    )

mu_2 = az.extract(preds_2, group="predictions", var_names="mu").to_numpy().T
y_2 = az.extract(preds_2, group="predictions", var_names="temp3pm").to_numpy().T
mu_media_2 = mu_2.mean(axis=0)
mu_inferior_2, mu_superior_2 = np.quantile(mu_2, [0.025, 0.975], axis=0)
y_inferior_2, y_superior_2 = np.quantile(y_2, [0.025, 0.975], axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for grupo, ciudad in zip(grupos, ciudades):
    tramo = slice(grupo * len(x_grid), (grupo + 1) * len(x_grid))
    seleccion = ciudad_idx == grupo
    ax.plot(x_grid, mu_media_2[tramo], color=f"C{grupo}")
    ax.fill_between(x_grid, y_inferior_2[tramo], y_superior_2[tramo], color=f"C{grupo}", alpha=0.15)
    ax.fill_between(
        x_grid, mu_inferior_2[tramo], mu_superior_2[tramo], color=f"C{grupo}", alpha=0.45
    )
    ax.scatter(x[seleccion], y[seleccion], color=f"C{grupo}", s=20, alpha=0.7, label=ciudad)

ax.set(xlabel="Temperatura a las 9 (°C)", ylabel="Temperatura a las 15 (°C)")
ax.legend(title="Ciudad");

Las rectas pasan por el centro de cada nube y las bandas predictivas se estrechan. Conocer la ciudad aporta información que el primer modelo dejaba afuera. Veamos si también mejora el chequeo predictivo.

In [ ]:
with modelo_2:
    pm.sample_posterior_predictive(
        idata_2, var_names=["temp3pm"], extend_inferencedata=True, random_seed=SEMILLA
    )

In [ ]:
az.plot_ppc_dist(idata_2, var_names="temp3pm", num_samples=50, figure_kwargs={"figsize": (7, 3.5)})
ax = plt.gcf().axes[0]
ax.set(
    title="Modelo 2: Un intercepto por ciudad",
    xlabel="Temperatura a las 15 (°C)",
    ylabel="Densidad",
);

## Modelo 3: interceptos y pendientes por ciudad

Agreguemos una pendiente para cada ciudad y veamos si esa flexibilidad aporta algo más.

In [ ]:
with pm.Model(coords=coords) as modelo_3:
    x_data = pm.Data("temp9am", x, dims="obs_idx")
    ciudad_data = pm.Data("ciudad_idx", ciudad_idx, dims="obs_idx")

    beta_0 = pm.Normal("beta_0", mu=10, sigma=5, dims="ciudad")
    beta_1 = pm.Normal("beta_1", mu=0, sigma=1, dims="ciudad")
    sigma = pm.HalfNormal("sigma", sigma=10)
    mu = pm.Deterministic("mu", beta_0[ciudad_data] + beta_1[ciudad_data] * x_data, dims="obs_idx")
    pm.Normal("temp3pm", mu=mu, sigma=sigma, observed=y, dims="obs_idx")

In [ ]:
with modelo_3:
    idata_3 = pm.sample(chains=4, random_seed=SEMILLA)

In [ ]:
az.summary(idata_3, var_names=parametros, ci_prob=0.95, round_to=2)

In [ ]:
with clone_model(modelo_3):
    pm.set_data(
        {"temp9am": x_grid_ciudades, "ciudad_idx": ciudad_grid},
        coords={"obs_idx": np.arange(len(x_grid_ciudades))},
    )
    preds_3 = pm.sample_posterior_predictive(
        idata_3, var_names=["mu", "temp3pm"], predictions=True, random_seed=SEMILLA
    )

mu_3 = az.extract(preds_3, group="predictions", var_names="mu").to_numpy().T
y_3 = az.extract(preds_3, group="predictions", var_names="temp3pm").to_numpy().T
mu_media_3 = mu_3.mean(axis=0)
mu_inferior_3, mu_superior_3 = np.quantile(mu_3, [0.025, 0.975], axis=0)
y_inferior_3, y_superior_3 = np.quantile(y_3, [0.025, 0.975], axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for grupo, ciudad in zip(grupos, ciudades):
    tramo = slice(grupo * len(x_grid), (grupo + 1) * len(x_grid))
    seleccion = ciudad_idx == grupo
    ax.plot(x_grid, mu_media_3[tramo], color=f"C{grupo}")
    ax.fill_between(x_grid, y_inferior_3[tramo], y_superior_3[tramo], color=f"C{grupo}", alpha=0.15)
    ax.fill_between(
        x_grid, mu_inferior_3[tramo], mu_superior_3[tramo], color=f"C{grupo}", alpha=0.45
    )
    ax.scatter(x[seleccion], y[seleccion], color=f"C{grupo}", s=20, alpha=0.7, label=ciudad)

ax.set(xlabel="Temperatura a las 9 (°C)", ylabel="Temperatura a las 15 (°C)")
ax.legend(title="Ciudad");

Las rectas y las bandas se parecen mucho a las del modelo anterior. Revisemos también las temperaturas que este modelo puede generar.

In [ ]:
with modelo_3:
    pm.sample_posterior_predictive(
        idata_3, var_names=["temp3pm"], extend_inferencedata=True, random_seed=SEMILLA
    )

In [ ]:
az.plot_ppc_dist(idata_3, var_names="temp3pm", num_samples=50, figure_kwargs={"figsize": (7, 3.5)})
ax = plt.gcf().axes[0]
ax.set(
    title="Modelo 3: Interceptos y pendientes por ciudad",
    xlabel="Temperatura a las 15 (°C)",
    ylabel="Densidad",
);

## Chequeos predictivos por ciudad

Comparemos las distribuciones por separado para cada ciudad. Cada fila corresponde a un modelo y cada columna a una ciudad. Las curvas azules representan 50 conjuntos de temperaturas simuladas y la negra, las temperaturas observadas. Usamos las mismas escalas en los seis paneles para facilitar la comparación.

In [ ]:
posteriors = [idata_1, idata_2, idata_3]
etiquetas = ["Una recta", "Intercepto por ciudad", "Intercepto y pendiente por ciudad"]

fig, axes = plt.subplots(3, 2, figsize=(10, 9), sharex=True, sharey=True, layout="constrained")
for fila, (idata, etiqueta) in enumerate(zip(posteriors, etiquetas)):
    replicas = az.extract(
        idata, group="posterior_predictive", var_names="temp3pm",
        num_samples=50, random_seed=SEMILLA,
    ).transpose("sample", "obs_idx").to_numpy()

    for columna, ciudad in enumerate(ciudades):
        ax = axes[fila, columna]
        seleccion = ciudad_idx == columna
        for replica in replicas[:, seleccion]:
            grilla, densidad, _ = az.kde(replica)
            ax.plot(grilla, densidad, color="C0", alpha=0.15)

        grilla, densidad, _ = az.kde(y[seleccion])
        ax.plot(grilla, densidad, color="black", lw=2)
        ax.set(
            title=f"{etiqueta}\n{ciudad}",
            xlabel="Temperatura a las 15 (°C)", ylabel="Densidad",
        )

## Reducción del desvío residual

El cambio importante aparece al distinguir las ciudades: el _posterior_ de $\sigma$ pasa de unos 4.1 a 2.4 °C. Permitir además pendientes diferentes apenas modifica ese desvío. Comparemos las tres distribuciones.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for idata, etiqueta in zip(posteriors, etiquetas):
    muestras = idata["posterior"]["sigma"].to_numpy().flatten()
    grilla, densidad, _ = az.kde(muestras)
    ax.plot(grilla, densidad, label=etiqueta, lw=2)
ax.set(xlabel=r"$\sigma$ (°C)", ylabel="Densidad", title="Desvío residual")
ax.legend();

In [ ]:
az.summary(idata_3, var_names="beta_1", kind="stats", ci_prob=0.95, round_to=2)

Acá tiene sentido conservar el modelo con un intercepto por ciudad y una pendiente común. El desvío residual baja de unos 4.1 a 2.4 °C al incorporar la ciudad. Las pendientes separadas quedan cerca de 0.87 y 0.81, y el tercer modelo casi no reduce ese desvío ni produce una mejora visible en estos chequeos.

Para este ejemplo, las pendientes distintas agregan complejidad con poco beneficio. Esto es una conclusión sobre estos datos y estas predicciones: las ciudades pueden diferir mucho en el nivel de la temperatura de la tarde y aun así tener asociaciones con la temperatura de la mañana muy parecidas.

La mejora tampoco significa que todos los días queden bien representados. Todavía hay observaciones alejadas de las rectas que merecerían una revisión.

## Respuestas

> ¿Cuál es la probabilidad de que la temperatura a la tarde sea mayor a 30 °C cuando a la mañana hicieron 22 °C en Uluru? ¿Y en Wollongong?

Usamos el modelo con interceptos distintos y obtenemos muestras predictivas para ambas ciudades cuando `temp9am` es 22 °C.

In [ ]:
with clone_model(modelo_2):
    pm.set_data({"temp9am": [22, 22], "ciudad_idx": [0, 1]}, coords={"obs_idx": [0, 1]})
    preds_22 = pm.sample_posterior_predictive(
        idata_2, var_names=["temp3pm"], predictions=True, random_seed=SEMILLA
    )

Seleccionamos las muestras para Uluru y calculamos la proporción que supera los 30 °C.

In [ ]:
(preds_22["predictions"].sel(obs_idx=0)["temp3pm"] > 30).mean().item()

Y hacemos lo mismo para Wollongong.

In [ ]:
(preds_22["predictions"].sel(obs_idx=1)["temp3pm"] > 30).mean().item()

En Uluru la probabilidad está alrededor de 0.54. En Wollongong es muy cercana a cero.

> Para cada ciudad, ¿cuál es la probabilidad de que la temperatura a la tarde sea mayor a 30 °C independientemente de la temperatura de la mañana?

Usamos las muestras predictivas que ya obtuvimos con las temperaturas de la mañana observadas. Así promediamos sobre los días de cada ciudad que tenemos en la muestra.

Primero, Uluru.

In [ ]:
y_predictive_uluru = idata_2["posterior_predictive"].sel(obs_idx=ciudad_idx == 0)["temp3pm"]
(y_predictive_uluru > 30).mean().item()

Luego, Wollongong.

In [ ]:
y_predictive_wollongong = idata_2["posterior_predictive"].sel(obs_idx=ciudad_idx == 1)["temp3pm"]
(y_predictive_wollongong > 30).mean().item()

Es bastante probable observar temperaturas mayores a 30 °C en Uluru, pero no así en Wollongong.

> Para cada ciudad, ¿cuál es la probabilidad de que la temperatura de la tarde sea menor que la de la mañana?

Ahora comparamos cada temperatura predictiva de la tarde con la temperatura observada a la mañana de ese día. No necesitamos volver a simular.

Empecemos por Uluru.

In [ ]:
x_observed_uluru = idata_2["constant_data"].sel(obs_idx=ciudad_idx == 0)["temp9am"]
(y_predictive_uluru < x_observed_uluru).mean().item()

En Uluru es muy poco frecuente. Veamos qué ocurre en Wollongong.

In [ ]:
x_observed_wollongong = idata_2["constant_data"].sel(obs_idx=ciudad_idx == 1)["temp9am"]
(y_predictive_wollongong < x_observed_wollongong).mean().item()

En Wollongong obtenemos una probabilidad cercana al 24%. Aunque la tarde suele ser más cálida que la mañana, también puede ocurrir lo contrario.